<a href="https://colab.research.google.com/github/marchedev2002/ia/blob/main/agente_inmobiliaria/agente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install -q langchain==0.3.7 langchain-core==0.3.15 langchain-community==0.3.5

SE PUEDE PROBAR CON OTRO MODELO

In [15]:
import os
import getpass

# Intentamos leer desde los Secretos de Colab (ícono de la llave); si no existe, la pide en pantalla
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Ingresa tu GROQ_API_KEY (empieza con gsk_...): ")

print("Clave de Groq configurada correctamente.")

Clave de Groq configurada correctamente.


## 1. Módulo Relacional: Base de Datos Transaccional (SQLite)
Almacenamos los datos estructurados del sistema de administración inmobiliaria: contratos registrados, fechas de vencimiento, montos de canon y seguimiento de morosidad.

In [16]:
import sqlite3

# Conexión local a la base SQLite en el entorno temporal de Colab
db_conn = sqlite3.connect("inmobiliaria_backoffice.db")
cursor = db_conn.cursor()

# 1. Tabla de contratos
cursor.execute("""
CREATE TABLE IF NOT EXISTS contratos (
    id_contrato TEXT PRIMARY KEY,
    direccion TEXT NOT NULL,
    inquilino_nombre TEXT NOT NULL,
    propietario_nombre TEXT NOT NULL,
    monto_canon REAL NOT NULL,
    fecha_inicio DATE NOT NULL,
    fecha_vencimiento DATE NOT NULL
);
""")

# 2. Tabla de cobros y mora
cursor.execute("""
CREATE TABLE IF NOT EXISTS pagos (
    id_pago INTEGER PRIMARY KEY AUTOINCREMENT,
    id_contrato TEXT NOT NULL,
    periodo TEXT NOT NULL,
    estado TEXT NOT NULL, -- 'AL_DIA', 'MORA'
    dias_atraso INTEGER DEFAULT 0,
    FOREIGN KEY(id_contrato) REFERENCES contratos(id_contrato)
);
""")

# Carga de datos de prueba representativos
contratos_mock = [
    ("LOC-2024-01", "Av. Corrientes 1240 4B", "Juan Perez", "Roberto Gomez", 350000.0, "2024-01-01", "2026-01-01"),
    ("LOC-2024-02", "Bv. Oroño 850 1A", "Lucia Fernandez", "Marta Lopez", 480000.0, "2024-03-01", "2026-03-01"),
    ("LOC-2024-03", "San Martin 520 PB", "Carlos Tevez", "Esteban Quito", 290000.0, "2023-11-01", "2025-11-01")
]

pagos_mock = [
    ("LOC-2024-01", "Septiembre 2024", "MORA", 12),
    ("LOC-2024-02", "Septiembre 2024", "AL_DIA", 0),
    ("LOC-2024-03", "Septiembre 2024", "MORA", 5)
]

cursor.executemany("INSERT OR REPLACE INTO contratos VALUES (?,?,?,?,?,?,?)", contratos_mock)
cursor.executemany("INSERT OR REPLACE INTO pagos (id_contrato, periodo, estado, dias_atraso) VALUES (?,?,?,?)", pagos_mock)
db_conn.commit()

print("Base de datos SQLite inicializada y cargada con éxito.")

Base de datos SQLite inicializada y cargada con éxito.


## 2. Módulo RAG: Indexación Vectorial de Contratos Legales (ChromaDB)
Procesamos las cláusulas legales no estructuradas de los contratos. Se aplica chunking semántico delimitado por cláusulas y vectorización con un modelo de embeddings de HuggingFace que se ejecuta localmente sin costo.

In [17]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Textos contractuales no estructurados
c1 = """
CONTRATO DE LOCACIÓN: LOC-2024-01 | Inmueble: Av. Corrientes 1240 4B
CLÁUSULA PRIMERA (DESTINO): Destino exclusivo vivienda familiar y permanente.
CLÁUSULA SEGUNDA (PAGO Y MORA): Vencimiento el día 10 de cada mes. La mora devengará un interés punitorio diario del 0.6% por cada día de retraso.
CLÁUSULA TERCERA (EXPENSAS Y REPARACIONES): Las expensas ordinarias corresponden al inquilino. Las extraordinarias y arreglos de cañerías maestras son a cargo del propietario.
CLÁUSULA CUARTA (RESCISIÓN ANTICIPADA): Requiere preaviso formal de 30 días e indemnización equivalente a 1 mes de canon de alquiler.
"""

c2 = """
CONTRATO DE LOCACIÓN: LOC-2024-02 | Inmueble: Bv. Oroño 850 1A
CLÁUSULA PRIMERA (DESTINO): Uso comercial o estudio profesional habilitado.
CLÁUSULA SEGUNDA (PAGO Y MORA): Vencimiento el día 5 de cada mes. Interés punitorio por mora pactado en 1.0% diario.
CLÁUSULA TERCERA (GARANTÍA): Fianza asumida solidariamente por Carlos Fernandez.
CLÁUSULA CUARTA (RESCISIÓN ANTICIPADA): Exige 60 días de preaviso y abono de 2 meses de canon indemnizatorio.
"""

c3 = """
CONTRATO DE LOCACIÓN: LOC-2024-03 | Inmueble: San Martin 520 PB
CLÁUSULA PRIMERA (DESTINO): Vivienda familiar sin admisión de sublocaciones.
CLÁUSULA SEGUNDA (PAGO Y MORA): Vencimiento el día 10. Tasa diaria punitoria por mora del 0.4% por día de retraso.
CLÁUSULA TERCERA (EXPENSAS): Expensas comunes por el locatario; fondos de reserva a cargo del locador.
"""

documentos = [
    Document(page_content=c1, metadata={"id_contrato": "LOC-2024-01", "direccion": "Av. Corrientes 1240 4B"}),
    Document(page_content=c2, metadata={"id_contrato": "LOC-2024-02", "direccion": "Bv. Oroño 850 1A"}),
    Document(page_content=c3, metadata={"id_contrato": "LOC-2024-03", "direccion": "San Martin 520 PB"})
]

# Segmentación semántica respetando delimitación de cláusulas
splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=50, separators=["CLÁUSULA ", "\n\n", "\n", " "])
chunks = splitter.split_documents(documentos)

# Embeddings gratuitos de HuggingFace en CPU
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings, collection_name="contratos_colab")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"RAG inicializado: {len(chunks)} fragmentos indexados en ChromaDB.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG inicializado: 4 fragmentos indexados en ChromaDB.


## 3. Catálogo de Herramientas (*Tools*) del Agente
Definimos las herramientas que permiten al agente interactuar con los distintos módulos:
1. `consultar_contrato_rag`: Búsqueda semántica documental de cláusulas legales.
2. `ejecutar_consulta_sql`: Motor de consulta relacional para datos de cobros y contratos.
3. `calcular_mora_exacta`: Cálculo aritmético preciso para liquidaciones financieras.
4. `redactar_intimacion`: Generación de notificaciones de cobranza.

In [18]:
def consultar_contrato_rag(consulta: str) -> str:
    """Busca cláusulas legales sobre mora, rescisión o expensas en ChromaDB."""
    docs = retriever.invoke(consulta)
    if not docs:
        return "No se hallaron cláusulas relevantes para esa búsqueda."
    return "\n---\n".join([f"[{d.metadata.get('id_contrato')}]: {d.page_content.strip()}" for d in docs])

def ejecutar_consulta_sql(query_sql: str) -> str:
    """Ejecuta sentencias SELECT sobre las tablas 'contratos' y 'pagos'."""
    if not query_sql.strip().upper().startswith("SELECT"):
        return "Error: Solo se autorizan consultas de tipo SELECT."
    try:
        cur = db_conn.cursor()
        cur.execute(query_sql)
        filas = cur.fetchall()
        columnas = [d[0] for d in cur.description]
        return f"Columnas: {columnas}\nFilas: {filas}"
    except Exception as e:
        return f"Error SQL: {str(e)}"

def calcular_mora_exacta(monto_canon: float, dias_mora: int, tasa_diaria_porcentaje: float) -> str:
    """Calcula con precisión matemática el interés acumulado y el total adeudado."""
    interes = monto_canon * (tasa_diaria_porcentaje / 100.0) * dias_mora
    total = monto_canon + interes
    return (
        f"Monto Base: ${monto_canon:,.2f} | Días de atraso: {dias_mora} | Tasa: {tasa_diaria_porcentaje}%\n"
        f"Interés Punitorio: ${interes:,.2f}\n"
        f"TOTAL A LIQUIDAR: ${total:,.2f}"
    )

def redactar_intimacion(destinatario: str, direccion: str, detalle_deuda: str) -> str:
    """Genera la plantilla formal de intimación de cobro."""
    return f"""
==================== NOTIFICACIÓN FORMAL DE INTIMACIÓN ====================
Destinatario: {destinatario}
Inmueble: {direccion}

Por medio de la presente intimamos a usted a cancelar en un plazo perentorio de 48 hs
la deuda devengada a la fecha, según el siguiente detalle:

{detalle_deuda}

Transcurrido el plazo sin regularización, se remitirán los antecedentes al departamento
jurídico para iniciar el cobro por vía ejecutiva.

Atentamente,
Departamento de Cobranzas y Asuntos Legales.
===========================================================================
"""

print("Funciones nativas listas sin dependencias de LangChain.")

Funciones nativas listas sin dependencias de LangChain.


## 4. Orquestación del Agente ReAct (Tool-Calling con LLaMA 3.1)
El modelo analiza la consulta del analista y decide qué herramientas invocar de manera autónoma para resolver la tarea.

In [19]:
import json
from groq import Groq

client = Groq()

# Mapeo directo a funciones estándar de Python
mapeo_herramientas = {
    "consultar_contrato_rag": lambda args: consultar_contrato_rag(args.get("consulta", "")),
    "ejecutar_consulta_sql": lambda args: ejecutar_consulta_sql(args.get("query_sql", "")),
    "calcular_mora_exacta": lambda args: calcular_mora_exacta(
        monto_canon=float(args.get("monto_canon", 0)),
        dias_mora=int(args.get("dias_mora", 0)),
        tasa_diaria_porcentaje=float(args.get("tasa_diaria_porcentaje", 0))
    ),
    "redactar_intimacion": lambda args: redactar_intimacion(
        destinatario=args.get("destinatario", ""),
        direccion=args.get("direccion", ""),
        detalle_deuda=args.get("detalle_deuda", "")
    )
}

# Esquema de Tool Calling para LLaMA 3
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "consultar_contrato_rag",
            "description": "Busca cláusulas legales sobre mora pactada, rescisión, garantías o responsabilidades de expensas.",
            "parameters": {
                "type": "object",
                "properties": {"consulta": {"type": "string", "description": "Texto o cláusula contractual a buscar"}},
                "required": ["consulta"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "ejecutar_consulta_sql",
            "description": "Ejecuta consultas SELECT sobre las tablas 'contratos' y 'pagos' para ver inquilinos, cánones y moras.",
            "parameters": {
                "type": "object",
                "properties": {"query_sql": {"type": "string", "description": "Sentencia SQL SELECT válida"}},
                "required": ["query_sql"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calcular_mora_exacta",
            "description": "Calcula matemáticamente el recargo por mora acumulada y el monto total exigible.",
            "parameters": {
                "type": "object",
                "properties": {
                    "monto_canon": {"type": "number", "description": "Monto base de canon de alquiler"},
                    "dias_mora": {"type": "integer", "description": "Días corridos de retraso"},
                    "tasa_diaria_porcentaje": {"type": "number", "description": "Porcentaje diario pactado (ej: 0.6 para 0.6%)"}
                },
                "required": ["monto_canon", "dias_mora", "tasa_diaria_porcentaje"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "redactar_intimacion",
            "description": "Redacta el borrador formal de notificación de deuda para el inquilino.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destinatario": {"type": "string", "description": "Nombre completo del inquilino"},
                    "direccion": {"type": "string", "description": "Dirección del inmueble"},
                    "detalle_deuda": {"type": "string", "description": "Desglose financiero del monto adeudado"}
                },
                "required": ["destinatario", "direccion", "detalle_deuda"],
            },
        },
    }
]

system_prompt = """Eres 'InmoOps Copilot', analista interno de administración inmobiliaria.
Directivas operativas:
1. Para inquilinos, cánones, deudas y fechas usa 'ejecutar_consulta_sql'.
2. Para cláusulas o porcentajes pactados de mora usa 'consultar_contrato_rag'.
3. Para montos finales de mora NUNCA calcules de memoria; usa 'calcular_mora_exacta'.
4. Si piden redactar una notificación formal usa 'redactar_intimacion'.
5. Sé claro y estructurado en tus respuestas.
6. NUNCA le expliques al usuario cómo ejecutar una consulta SQL él mismo ni le sugieras
   herramientas externas (pgAdmin, MySQL Workbench, etc.). Vos tenés acceso directo a
   las herramientas: siempre debés invocarlas y responder con los resultados reales
   obtenidos, nunca con instrucciones para que el usuario los obtenga por su cuenta."""

def ejecutar_agente(pregunta_usuario: str):
    mensajes = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": pregunta_usuario}
    ]

    while True:
        respuesta = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=mensajes,
            tools=tools_schema,
            tool_choice="auto",
            temperature=0
        )
        msg = respuesta.choices[0].message
        mensajes.append(msg)

        # Si el modelo terminó de razonar y no llama herramientas, entrega respuesta final
        if not msg.tool_calls:
            return msg.content

        # Ejecución y devolución de resultados de las tools
        for llamada in msg.tool_calls:
            nombre = llamada.function.name
            args = json.loads(llamada.function.arguments)
            resultado = mapeo_herramientas[nombre](args)

            mensajes.append({
                "role": "tool",
                "tool_call_id": llamada.id,
                "name": nombre,
                "content": str(resultado)
            })

print("Agente InmoOps Copilot compilado y listo.")

Agente InmoOps Copilot compilado y listo.


## 5. Demostración Operativa y Casos de Prueba
Se ejecutan tres pruebas para validar el comportamiento del sistema experto:
1. Búsqueda contextual puramente documental (RAG).
2. Extracción analítica estructurada (SQL).
3. Pipeline agéntico integral (SQL -> RAG -> Math Tool -> Drafting Tool).

In [20]:
res_1 = ejecutar_agente("¿A cargo de quién corresponden las reparaciones de cañerías y las expensas extraordinarias en el contrato de Av. Corrientes 1240?")
print(res_1)

En el contrato de **Av. Corrientes 1240** se establece lo siguiente:

| Concepto | Responsabilidad |
|----------|-----------------|
| **Reparaciones de cañerías** | **Propietario** (el arrendador) |
| **Expensas extraordinarias** | **Propietario** (el arrendador) |

> **Cláusula Tercera (Expensas y Reparaciones)**: “Las expensas ordinarias corresponden al inquilino. **Las extraordinarias y arreglos de cañerías maestras son a cargo del propietario**.”

Por lo tanto, tanto las reparaciones de cañerías como las expensas extraordinarias son responsabilidad del propietario, no del inquilino.


In [21]:
res_2 = ejecutar_agente("¿Qué contratos registran pagos en estado de MORA en el sistema y cuántos días de retraso tiene cada uno?")
print(res_2)

**Contratos con pagos en estado de MORA**

| Contrato | Días de retraso |
|----------|-----------------|
| LOC‑2024‑01 | 12 días |
| LOC‑2024‑03 | 5 días |

Estos son los contratos que, según el registro de pagos, tienen al menos un pago con estado “MORA” y el número de días de retraso correspondiente a cada uno.


In [22]:
res_3 = ejecutar_agente(
    "Revisa si el contrato de Av. Corrientes 1240 tiene pagos pendientes este mes. "
    "Si está en mora, consulta en su contrato qué tasa de interés diario se pactó, "
    "calcula el total adeudado y déjame preparado el borrador de intimación formal."
)
print(res_3)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m2984wche5qa8f8r2vfkmedd` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 3210, Requested 5529. Please try again in 5.5425s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## 6. Documentación Técnica y Relación con TP1 (Criterios de Evaluación)

### Decisiones de Diseño
1. **Arquitectura Híbrida (RAG + SQL):** Se delimitó qué información es estructurada/transaccional (SQLite) y cuál interpretativa/documental (ChromaDB), reduciendo alucinaciones en fechas y saldos.
2. **Tools Determinísticas:** El LLM delega las operaciones aritméticas a código Python nativo para asegurar precisión contable en las liquidaciones de mora.
3. **Chunking Especializado:** Se aplicó segmentación respetando palabras clave (`CLÁUSULA`) para evitar fragmentar definiciones legales entre chunks contiguos.

### Vinculación con TP1 (Redes Neuronales)
- **TP1:** Se resolvieron problemas de clasificación mediante una red neuronal supervisada que optimizó pesos fijos mediante backpropagation sobre features tabulares.
- **TP2:** Se utiliza un modelo fundacional preentrenado (LLM) asistido por representaciones semánticas densas (embeddings) y orquestación dinámica de herramientas externas en tiempo de ejecución.